# Reconstruct mesh from point cloud with meshlab

In [32]:
from pathlib import Path

def resolve_npz(data_root: Path, object_id: str | None, explicit: Path | None) -> Path:
    """Magic function to find your npz file with just object_id.
    
    Expect:
    ```
    object_id/
        |__affordance.npz
    ```
    """
    if not data_root.exists():
        raise ValueError("Invalid data root path!")
    if explicit is not None:
        path = explicit.resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    if not object_id:
        raise ValueError("Supply --object or --npz")

    matches = list(data_root.rglob(f"{object_id}/affordance.npz")) # search directly under data/
    if not len(matches): # between data and object id there is something else
        matches = list(data_root.rglob(f"*/{object_id}/affordance.npz"))
        if not len(matches):
            raise ValueError(f"Unable to resolve {object_id}/affordance.npz. Use --npz")
    
    return matches[0]

In [38]:
from pathlib import Path
from scipy.spatial import cKDTree
import numpy as np
import pymeshlab
# from generate_graspnet_grasp import resolve_npz
import os

DATA_ROOT = Path('data/laso_grasp_only_affordance').resolve()
npz_path = resolve_npz(DATA_ROOT, "mug/1bc5d303ff4d6e7e1113901b72a68e7c", None)
OUT_DIR = Path("outputs/reconstructed_mesh")
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
out_path = os.path.join(OUT_DIR, 'mesh.ply')

with np.load(npz_path) as data:
    points = np.asarray(data["points"], dtype=np.float64)
    normals = np.asarray(data["normals"], dtype=np.float64)

cloud = pymeshlab.Mesh(
    vertex_matrix=points,
    v_normals_matrix=normals,
)

meshes = pymeshlab.MeshSet()
meshes.add_mesh(cloud, "point_cloud")

# LASO already provides normals.
# Recompute them only if the stored normals are unreliable.
# meshes.compute_normal_for_point_clouds(
#     k=20,
#     smoothiter=2,
# )



tree = cKDTree(points)

distances, _ = tree.query(points, k=2)

# Column zero is each point's distance to itself.
nearest_distances = distances[:, 1]

median_spacing = float(np.median(nearest_distances))
bbox_diagonal = float(np.linalg.norm(np.ptp(points, axis=0)))

alpha = 2.0

radius_percent = (
    100.0 * alpha * median_spacing / bbox_diagonal
)

print("median spacing:", median_spacing)
print("bounding-box diagonal:", bbox_diagonal)
print("suggested radius percentage:", radius_percent)

meshes.generate_surface_reconstruction_ball_pivoting(
    ballradius=pymeshlab.PercentageValue(radius_percent),
)
# radius_percentages = [1.5, 2.0, 2.5, 3.5]

# for index, radius in enumerate(radius_percentages):
#     meshes.generate_surface_reconstruction_ball_pivoting(
#         ballradius=pymeshlab.PercentageValue(radius),
#         deletefaces=index == 0,
#     )

# meshes.generate_surface_reconstruction_ball_pivoting(
#     # ballradius=pymeshlab.PercentageValue(2.5),
# )

# meshes.meshing_close_holes(
#     maxholesize=20,
#     selected=False,
#     newfaceselected=True,
#     selfintersection=True,
#     refinehole=True,
# )

# meshes.apply_coord_taubin_smoothing(
#     stepsmoothnum=10,
#     lambda_=0.5,
#     mu=-0.53,
# )

meshes.compute_normal_per_vertex()

meshes.save_current_mesh(str(out_path))

median spacing: 0.06087141077684127
bounding-box diagonal: 2.3268489051234162
suggested radius percentage: 5.232089685136874


In [39]:
from plotly import graph_objects 

mesh = meshes.current_mesh()

vertices = mesh.vertex_matrix()
faces = mesh.face_matrix()

print(f"vertices: {len(vertices)}")
print(f"faces: {len(faces)}")

if len(faces) == 0:
    raise RuntimeError("Surface reconstruction produced no triangle faces")

figure = graph_objects.Figure(
    data=[
        graph_objects.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1],
            z=vertices[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            intensity=vertices[:, 2],
            colorscale="Viridis",
            flatshading=True,
            opacity=1.0,
            name="Reconstructed mesh",
        )
    ]
)

figure.update_layout(
    title="Ball-Pivoting Reconstruction",
    width=900,
    height=750,
    scene=dict(
        aspectmode="data",
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
    ),
)

figure.show()

vertices: 2048
faces: 4003


In [40]:
import trimesh

mesh = trimesh.load(out_path, force="mesh", process=False)
components = mesh.split(only_watertight=False)

print("vertices:", len(mesh.vertices))
print("faces:", len(mesh.faces))
print("connected components:", len(components))
print("watertight:", mesh.is_watertight)

vertices: 2048
faces: 4003
connected components: 1
watertight: False
